# AgriSwin-MoE: train on Colab

Runs the **same** `train.py`, recipe and frozen splits as the laptop. Nothing
about training lives in this notebook: it prepares the runtime and starts the
queue runner.

The pipeline is identical everywhere, but the GPU is not a detail: the same
seed on a different GPU gives a different model (measured: Swin-V2-T seed 42
was one test image and 0.05 F1 points apart on the laptop and the L4). So each
comparison table is trained on **one** machine. The Phase 3 control was trained
on the L4, so every run compared against it must be too.

**Before you run anything**

1. **Runtime > Change runtime type > L4 GPU** > Save. Not the A100 for Phase 3:
   it would be a fresh draw against an L4 control.
2. *Recommended:* add a `GITHUB_TOKEN` secret (the key icon in the left sidebar)
   so each result is pushed to GitHub the moment its run finishes. Without it,
   results are kept on your Google Drive. How to make the token:
   `docs/NEXT_STEPS.md`, Part H2.
3. **Runtime > Run all**, and keep this tab open while it trains, with the
   computer awake and on the charger.

**If the runtime disconnects**, open this notebook again and *Run all*: the
queue field defaults to the queue being trained, finished runs are skipped, and
an interrupted one resumes from its last finished epoch, because checkpoints
live on your Drive.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Connect Google Drive. Checkpoints and the image cache live there, so they
#    survive a disconnect. The first time, Colab asks you to allow access.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Get the latest code. In a runtime that has already trained, --rebase
#    keeps any result commits made here on top of the newest code.
import os
REPO = "/content/agriswin-moe"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} pull --rebase
else:
    !git clone https://github.com/amritaj312-max/agriswin-moe.git {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Prepare this runtime: exact packages, images verified byte for byte
#    against the laptop's, checkpoints on Drive. The first time this downloads
#    the images (a few minutes) and caches them on Drive; after that it copies
#    the cache (about a minute).
import importlib
import sys
sys.path.insert(0, f"{REPO}/notebooks")
import colab_runtime
importlib.reload(colab_runtime)
runtime = colab_runtime.setup()

## What to train

A queue file holds one `train.py` argument line per run, and the field below
defaults to the one being trained now. The queues so far, in order:

| Queue | What it trains | Status |
|---|---|---|
| `colab_first.txt` | EfficientNet-B0 seed 42, Swin-V2-T seeds 1 and 2 | done |
| `colab_control.txt` | Swin-V2-T seed 42: completes the three-seed control | done |
| `colab_stdam.txt` | ST-DAM, full module, seeds 1, 2 and 42 (about 11 h) | **current** |

Each ST-DAM run is paired with the control run of the same seed: it starts
from the identical function, sees the same images in the same order, and
differs only in ST-DAM.

**Do not change the batch size or learning rate** for the bigger GPU: every row
of a table must share one recipe, and `train.py` refuses a paper run that
changes it. The speed-up comes from the GPU alone.

In [ ]:
# 5. Choose the queue. The default is the one being trained now, so that
#    reopening this notebook after a disconnect resumes it with Run all.
QUEUE = "ml_pipeline/configs/queues/colab_stdam.txt"  #@param {type:"string"}
!cat {QUEUE}

In [ ]:
# 6. Train. Each result is committed, and pushed with the token, as soon as
#    its run finishes. The img/s figure in the progress lines is this GPU's
#    real speed on this recipe.
NUM_WORKERS = runtime["num_workers"]
TRAILER = f"Trained on Colab ({runtime['gpu']})"
!python ml_pipeline/src/training/run_queue.py {QUEUE} --set-all num_workers={NUM_WORKERS} --commit-trailer "{TRAILER}"

In [ ]:
# 7. Every result so far, from every machine. The Splits and Recipe lines at
#    the bottom must say all runs match.
!python ml_pipeline/src/evaluation/summarize_results.py